<a href="https://colab.research.google.com/github/hamshini1413/deep-learning/blob/main/RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
!pip install datasets


In [16]:
# ==========================================
# Step 1: Install and Load the Dataset
# ==========================================

# Run this command in terminal or Colab:
# pip install datasets tensorflow

from datasets import load_dataset
import re
import numpy as np

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

In [17]:
# Load AG News dataset
raw_dataset=load_dataset("wangrongsheng/ag_news")
df = pd.DataFrame(raw_dataset["train"])
print(df.head())

                                                text  label
0  Wall St. Bears Claw Back Into the Black (Reute...      2
1  Carlyle Looks Toward Commercial Aerospace (Reu...      2
2  Oil and Economy Cloud Stocks' Outlook (Reuters...      2
3  Iraq Halts Oil Exports from Main Southern Pipe...      2
4  Oil prices soar to all-time record, posing new...      2


In [18]:
# Extract text and labels
texts = train_data["text"]
labels = train_data["label"]

print("Number of samples:", len(texts))

Number of samples: 120000


In [19]:
# ==========================================
# Step 2: Clean and Normalize the Text
# ==========================================

def clean_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove punctuation, numbers, and special symbols
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

cleaned_texts = [clean_text(text) for text in texts]

print(cleaned_texts[0])


wall st bears claw back into the black reuters reuters shortsellers wall streets dwindlingband of ultracynics are seeing green again


In [20]:
# ==========================================
# Step 3: Tokenize the Strings into Numbers
# ==========================================
VOCAB_SIZE = 20000

tokenizer = Tokenizer(num_words=VOCAB_SIZE)

tokenizer.fit_on_texts(cleaned_texts)

sequences = tokenizer.texts_to_sequences(cleaned_texts)

print(sequences[0])

[391, 324, 1525, 14260, 99, 54, 1, 812, 23, 23, 391, 1988, 4, 34, 3893, 737, 295]


In [21]:
# ==========================================
# Step 4: Pad and Truncate Sequences
# ==========================================
MAX_LENGTH = 50

X = pad_sequences(
    sequences,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

print(X.shape)

(120000, 50)


In [22]:
# ==========================================
# Step 5: Convert Labels to Categorical
# ==========================================

num_classes = 4

y = to_categorical(labels, num_classes=num_classes)

print(y[0])


[0. 0. 1. 0.]


In [23]:
# ==========================================
# Split Training and Validation Data
# ==========================================

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [27]:
# ==========================================
# Step 6: Define the RNN Model
# ==========================================

model = Sequential([

    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128
    ),

    SimpleRNN(64),

    Dense(
        num_classes,
        activation="softmax"
    )
])

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [25]:
# ==========================================
# Step 7: Compile and Train the Model
# ==========================================
model.compile(

    optimizer="adam",

    loss="categorical_crossentropy",

    metrics=["accuracy"]
)

history = model.fit(

    X_train,
    y_train,

    epochs=5,

    batch_size=64,

    validation_data=(X_val, y_val)
)


Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 40s 26ms/step - accuracy: 0.8094 - loss: 0.5416 - val_accuracy: 0.8855 - val_loss: 0.3754
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 39s 25ms/step - accuracy: 0.9035 - loss: 0.3236 - val_accuracy: 0.8866 - val_loss: 0.3649
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 41s 25ms/step - accuracy: 0.9179 - loss: 0.2739 - val_accuracy: 0.8931 - val_loss: 0.3573
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 36s 24ms/step - accuracy: 0.9276 - loss: 0.2447 - val_accuracy: 0.8925 - val_loss: 0.3913
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 37s 25ms/step - accuracy: 0.9315 - loss: 0.2321 - val_accuracy: 0.8881 - val_loss: 0.3858


In [26]:
# ==========================================
# Step 8: Evaluate Accuracy
# ==========================================

loss, accuracy = model.evaluate(
    X_val,
    y_val
)

print("\nValidation Accuracy:", accuracy)
print("Validation Loss:", loss)

750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8881 - loss: 0.3858

Validation Accuracy: 0.8880833387374878
Validation Loss: 0.3858204782009125
